# Weather Impact Analysis

How weather conditions affect `arrival_delay`: rain, heavy rain, wind, snow and temperature.

## Setup

In [ ]:
import polars as pl
from pathlib import Path

from wgnd.core.theme import setup
from wgnd.core._output import section_header, log, success

from zh_tram_flow.config import PATHS
from zh_tram_flow.settings import setup_plotting, logger

setup_plotting()
setup()
logger.info("notebook started")

TRAIN = PATHS["processed"] / "train_features.parquet"
TEST  = PATHS["processed"] / "test_features.parquet"

lf = pl.scan_parquet(TRAIN)

## Rain

Delay comparison: `has_rain = True` vs `False`. Baseline weather impact.

In [ ]:
section_header("Rain Impact")

rain = (
    lf.group_by("has_rain")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("has_rain")
    .collect()
)

## Heavy Rain

Does heavy rain (`precipitation > 5mm`) significantly increase delays beyond normal rain?

In [ ]:
section_header("Heavy Rain Impact")

heavy_rain = (
    lf.group_by("has_heavy_rain")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("has_heavy_rain")
    .collect()
)

## Wind

Impact of high wind (`wind_speed > 40 km/h`) on delay.

In [ ]:
section_header("Wind Impact")

wind = (
    lf.group_by("is_windy")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("is_windy")
    .collect()
)

## Snow

Snow conditions (`precipitation > 0 & temperature < 2°C`) — likely strongest weather effect.

In [ ]:
section_header("Snow Impact")

snow = (
    lf.group_by("has_snow")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("has_snow")
    .collect()
)

## Temperature

Continuous temperature effect on delay — linear correlation and binned analysis.

In [ ]:
section_header("Temperature Correlation")

temp_bins = (
    lf.with_columns(
        (pl.col("temperature") / 5).floor().cast(pl.Int8).alias("temp_bin_5c")
    )
    .group_by("temp_bin_5c")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("temp_bin_5c")
    .collect()
)